# 🎯 [K12_M03] 챗봇 역할 프롬프트 작성하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m03_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 03. 챗봇 역할 프롬프트 작성하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 03. 챗봇 역할 프롬프트 작성하기
**소요시간**: 약 1.5시간

![미션 3 대대 행정 챗봇 단일 역할 페르소나 및 4대 행동 제약 프롬프트 아키텍처 다이어그램](/uploads/common/K12_m03_system_prompt_role.png)

### 📋 미션 개요 및 배경
생성형 AI가 부대 행정 도우미로서의 정체성을 확립하고 제공된 FAQ 자료에만 엄격히 근거하여 답하도록 통제하는 시스템 프롬프트(지시문)를 설계합니다. '가상 한빛대대 행정 업무 안내 챗봇' 페르소나, 정중한 어조, 개인정보 수집 금지, FAQ 외 내용 답변 금지 규칙을 수립하고 가상 FAQ 데이터를 컨텍스트로 주입합니다. 페르소나 및 제약 지시 유무에 따른 답변 품질과 안전성을 실증 비교하여 프롬프트 역할 설계 원리를 이해합니다.

### 🎯 미션 학습목표
'너는 가상 한빛대대 행정 업무 안내 챗봇이다'로 역할을 주고, FAQ만 근거로 답하기, 존댓말, 개인정보 묻지 않기 같은 규칙과 FAQ 표를 함께 넣습니다. 질문 카드 5개로 시험하고 역할 지시가 없을 때와 비교합니다. **이를 통해 페르소나 부여(Role-Playing)와 컨텍스트 주입(In-Context Learning)을 결합하여 AI의 응답 태도와 범위를 정밀 통제하는 시스템 프롬프트 엔지니어링 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 역할 한 줄과 규칙 4가지를 쓴다.
#### 이 단계의 역할

챗봇 페르소나('가상 한빛대대 행정 업무 안내 챗봇')와 4대 필수 지시 규칙(FAQ 근거 답변, 정중한 경어체, 개인정보 수집 금지, 모르는 내용은 추측 금지)을 설계합니다.

#### 해야 할 일

- 챗봇의 명확한 역할 정의 문장 1줄을 작성합니다.

- 답변의 신뢰성과 안전성을 보장하는 4대 필수 준수 규칙을 불릿으로 구조화합니다.

- AI에게 프롬프트의 지시 문구가 명확하고 엄격한지 검토를 요청합니다.

#### 💡 AI 프롬프트 질의 예시

> **[시스템 프롬프트 역할 및 제약 규칙 설계]:** "너는 군 특화 AI 프롬프트 엔지니어링 전문가야. '가상 한빛대대 행정 업무 안내 챗봇'을 위한 시스템 프롬프트(지시문)를 작성해 줘. [역할 정의 1줄, 필수 준수 규칙 4가지(FAQ 근거 제한, 군사보안/개인정보 보호, 말투, 사실 검증)]를 체계적인 구조로 구성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **4대 원칙 반영:** 1) FAQ 외 추측 금지, 2) 정중한 존댓말, 3) 개인정보(군번/주민번호) 질문 금지, 4) 보안 수칙 준수가 모두 포함되었는가?
- **지시 강도:** 모호한 권유형('~하면 좋습니다')이 아닌 엄격한 명령형('~하십시오', '~하지 마십시오')으로 작성되었는가?

#### 핵심 확인 지표

> 역할 1줄 및 4대 제약 규칙 초안

#### 학습자가 제출해야 하는 내용

- 시스템 프롬프트 규칙 초안

#### 🔹 Step 2. FAQ 표를 붙여 넣는다.
#### 이 단계의 역할

시스템 지시문 하단에 미션2에서 완성한 20개 행정 FAQ 표준 테이블을 컨텍스트(Context Injection)로 결합합니다.

#### 해야 할 일

- 시스템 지시문 아래에 명확한 구분자(--- 또는 ### [참조 행정 FAQ 데이터])를 배치합니다.

- 미션2의 FAQ 표 전문을 마크다운 형식 그대로 정확히 삽입합니다.

- "아래 표의 내용만 유일한 근거로 활용하여 답변하라"는 앵커링 지시문을 확인합니다.

#### 💡 AI 프롬프트 질의 예시

> **[컨텍스트 결합 프롬프트 구조화]:** "시스템 지시문과 참조 FAQ 표를 AI 모델이 가장 혼동 없이 분리 인식할 수 있도록 구분자와 헤더를 활용하여 프롬프트를 깔끔하게 결합해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **표 데이터 무결성:** 붙여 넣은 FAQ 표의 20개 행이 깨지거나 잘려 나간 곳이 없는가?
- **컨텍스트 참조 지시:** 표 데이터를 참조하라는 문구가 시스템 규칙에 명확히 결부되었는가?

#### 핵심 확인 지표

> 지시문과 FAQ 표가 결합된 통합 시스템 프롬프트

#### 학습자가 제출해야 하는 내용

- 통합 시스템 프롬프트 초안 전문

#### 🔹 Step 3. 질문 카드 5개로 시험한다.
#### 이 단계의 역할

역할 지시문과 FAQ가 주입된 챗봇 프롬프트에 미션1에서 선별한 우선 질문 5개를 입력하여 응답 결과를 수집합니다.

#### 해야 할 일

- 완성된 시스템 프롬프트를 AI 도구에 입력하여 대화 세션을 시작합니다.

- 우선 질문 5개를 순차적으로 질의하고 각각의 답변 전문을 기록합니다.

- 답변이 정중한 톤으로 FAQ의 단계별 절차를 정확히 따르는지 1차 점검합니다.

#### 💡 AI 프롬프트 질의 예시

> <p>**[챗봇 시험 질의 실행]:** "{통합 시스템 프롬프트 전문}

[장병 질문]: 포상휴가 신청하려면 언제까지 서류를 제출해야 하나요?"</p>

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **근거 준수:** FAQ에 명시된 접수 기한(출타 3일 전 등)과 구비 서류가 정확히 답변에 반영되었는가?
- **말투 일관성:** 정중한 군 행정 안내 어조가 유지되었는가?

#### 핵심 확인 지표

> 우선 질문 5개에 대한 챗봇 응답 기록표

#### 학습자가 제출해야 하는 내용

- 우선 질문 5개 질의 응답 로그

#### 🔹 Step 4. 역할 지시 없이 같은 질문을 해 비교한다.
#### 이 단계의 역할

시스템 프롬프트(역할/제약) 없이 동일한 질문 5개를 질의하여, AI 자체 기본 지식 기반 답변과 프롬프트 제어 답변의 정확도 및 안전성을 비교합니다.

#### 해야 할 일

- 시스템 지시문이나 FAQ 없이, 새로운 창에서 동일한 우선 질문 5개를 그대로 질의합니다.

- AI가 자체 지식으로 답변한 결과와 역할 프롬프트가 적용된 결과를 나란히 배치합니다.

- 답변 속 날짜, 절차, 서류의 왜곡 및 타 부대 규정 혼용 사례를 대조 분석합니다.

#### 💡 AI 프롬프트 질의 예시

> <p>**[역할 미부여 기본 질의]:** "(시스템 지시문 없이 순수 질문만 입력)
포상휴가 신청하려면 언제까지 서류를 제출해야 하나요?"</p>

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 식별:** 역할 지시가 없을 때 가상 한빛대대 규정이 아닌 일반 법령이나 임의의 일수를 지어내는가?
- **제약의 효과성:** 프롬프트 적용 시 거짓 답변(할루시네이션)이 얼마나 성공적으로 억제되었는가?

#### 핵심 확인 지표

> 역할 프롬프트 유무에 따른 5개 질문 비교표

#### 학습자가 제출해야 하는 내용

- 역할 유무 비교 분석표

#### 🔹 Step 5. 심화 도전 과제 및 최적화
#### 이 단계의 역할

장병이 초조하거나 급한 어조로 질문했을 때도 안정적으로 감정을 완화하고 차분하게 행정 절차를 안내하는 응대 지침을 추가합니다.

#### 해야 할 일

- "내일 당장 휴가 나가야 하는데 어떡해요!"와 같은 감정적/긴급 질문을 가정합니다.

- 공감적 경청과 함께 당직실 및 지휘계통 보고를 차분히 권고하는 감성 지침을 프롬프트에 추가합니다.

#### 💡 AI 프롬프트 질의 예시

> **[긴급 상황 감정 케어 지시문 고도화]:** "장병이 당황하거나 긴박한 어조로 질문할 때 안심시키는 어조로 규정된 긴급 절차와 당직실 문의처를 차분히 안내하도록 프롬프트 어조 규칙을 보강해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **규정 준수:** 감정적 대응에 치우쳐 부대 규정이나 절차를 생략하지 않는지 확인

#### 핵심 확인 지표

> 감성 대응 지침이 보강된 프롬프트 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **행정 FAQ 챗봇 시스템 프롬프트(지시문) 표준 템플릿 (MD):** 역할 정의(Role), 컨텍스트(Context), 4대 필수 지시 규칙(Constraints) 모듈형 뼈대 서식
- **우선 질문 5선 테스트 시트 및 질의응답 기록부:** 5개 핵심 질문별 질의 실행 및 AI 원시 응답 추출 양식
- **[학습 가이드] 역할 부여(Role-Playing)와 검색 증강(RAG) 프롬프트 엔지니어링:** 모델의 자체 사전 지식을 억제하고 주입된 FAQ 표에만 근거하여 답하게 하는 지시문 설계 원리

**실습 팁:** 프롬프트 상단에 "너는 가상 한빛대대 행정 업무 안내 챗봇이다. 반드시 아래 [참조 행정 FAQ]에 근거하여 답하고, 표에 없는 내용은 절대 추측하지 말라"는 명확한 앵커링 문구를 배치하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(역할 지정) — 행정 챗봇 페르소나와 4대 준수 규칙을 설계하고, 역할 부여 유무에 따른 답변 차이를 실증 비교했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 시스템 프롬프트(역할 1줄, 4대 규칙, FAQ 표 주입) 완성

- 5개 우선 질문 질의 실행 로그 수집 완료

- 역할 프롬프트 적용 전후 비교표(환각 억제 분석) 완성

#### 채점자 판정 메모

> 역할 지시가 없을 때 AI가 임의로 지어낸 답변(타 부대 규정, 임의 일수 등)을 정확히 지적하고 프롬프트의 통제 효과를 입증했는지 확인한다.



In [ ]:
# [미션 03.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_03_prompt = """
[대대 행정 업무 챗봇 시스템 프롬프트(지시문) 설계]
당신은 대한민국 육군 '가상 한빛대대 행정 업무 안내 챗봇'을 설계하는 AI 프롬프트 엔지니어입니다.
챗봇이 부대 행정 도우미로서 정체성을 유지하고 허위 정보(환각)를 방어할 수 있도록 다음 4대 제약 규칙이 포함된 시스템 프롬프트를 작성해 주십시오:
1. 근거 기반 답변 원칙 (제공된 FAQ 테이블 내용 외 임의 추측 금지)
2. 정중한 군 경어체 준수 ('~하십시오', '~입니다')
3. 군사보안 및 개인식별정보 보호 (군번, 주민번호 등 민감 정보 요구 금지)
4. 교육용 가상 부대 기준 및 인간 책임 원칙(HITL) 고지

또한 시스템 프롬프트(역할 부여) 적용 전과 후의 응답 품질 차이를 비교할 수 있는 3개 질문에 대한 질의 비교표를 작성해 주십시오.
"""

mission_03_result = """
## 1. 시스템 프롬프트 초안 (System Prompt)

```markdown
당신은 대한민국 육군 '가상 한빛대대 행정 업무 안내 챗봇'입니다.
장병들의 행정 업무(휴가, 증명서, 전입전출, 자기개발, 계정, 시설수리 등) 문의에 대해 친절하고 정확하게 안내하는 임무를 수행합니다.

[기본 준수 규칙]
1. [근거 기반 답변]: 반드시 하단의 [참조 행정 FAQ 데이터]에 수록된 내용만을 바탕으로 답변하십시오. 표에 없는 내용은 절대 임의로 추측하거나 지어내지 마십시오.
2. [어조 및 태도]: 항상 정중하고 명확한 군 행정 경어체('~하십시오', '~입니다')를 사용하십시오.
3. [보안 및 개인정보 보호]: 장병의 군번, 주민등록번호, 실제 부대 세부 위치 등 민감 개인정보를 요구하지 마십시오.
4. [책임 및 한계]: 안내하는 모든 행정 절차는 교육용 가상 부대 기준이며, 실제 최종 승인은 지휘계통을 따른다는 점을 염두에 두고 명확한 절차 단계와 서류를 안내하십시오.

### [참조 행정 FAQ 데이터]
(여기에 미션 2에서 완성한 20개 행정 FAQ 마크다운 표 전문 삽입)
```

---

## 2. 역할 부여 유무에 따른 질의 결과 비교표

| 질문 | 역할 미부여(기본 AI 응답) | 역할 프롬프트 적용 응답 | 비교 분석 (프롬프트 효과) |
| :--- | :--- | :--- | :--- |
| **Q1. 포상휴가 신청 기한** | "통상 군대에서 포상휴가는 1~2주 전에 중대장에게 구두 보고 후 신청합니다." | "포상휴가는 수여일로부터 6개월 이내 사용해야 하며, 출타 3일 전까지 포상명령 사본과 휴가신청서를 대대 인사과에 제출해야 합니다." | 일반적 추측 답변 대신 한빛대대 규정(6개월, D-3, 인사과)을 정확히 인용함 |
| **Q2. 복무확인서 발급처** | "행정반이나 국방복지포털, 정부24에서 온라인 신청이 가능할 수 있습니다." | "국방인사정보체계 또는 행정반에서 구두 신청하며, 대대 인사과에서 확인 후 24시간 이내 발급됩니다." | 모호한 외부 사이트 언급을 차단하고 부대 내 실제 발급 경로(인사과, 24시간)만 명시함 |
| **Q3. 자격증 지원 한도** | "군인공제회나 국방부 지원금은 보통 연간 10~20만 원 수준입니다." | "국가기술자격증 응시료 지원금은 1인당 연간 최대 12만 원 한도이며 대대 정훈공보실에서 주관합니다." | 부정확한 수치 범위를 배제하고 규정된 정확한 상한액(12만 원)을 못 박음 |
"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 03. 챗봇 역할 프롬프트 작성하기", mission_03_prompt, mission_03_result)

